In [0]:
from pyspark.sql import functions as F

In [0]:

CATALOG = "ecommerce_catalog"
VOLUME_ROOT = f"/Volumes/{CATALOG}/bronze/raw_files"
CHECKPOINT_ROOT = f"/Volumes/{CATALOG}/bronze/_checkpoints"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS ecommerce_catalog.bronze.raw_files")
spark.sql(f"CREATE VOLUME IF NOT EXISTS ecommerce_catalog.bronze._checkpoints")

#/Volumes/ecommerce_catalog/bronze/raw_files/full_load/parquet/

In [0]:
def bronze_autoload(source_path, table_name):
    df = (
        spark.read.option("header", True)
        .option("inferSchema", True)
        .parquet(source_path)
        .select("*", "_metadata.file_name", "_metadata.file_size")
    )

    df.write.mode("overwrite").format("delta")\
        .option("overweiteSchema", True)\
            .saveAsTable(f"ecommerce_catalog.bronze.{table_name}")
    
    print(table_name, df.count())


#products - /full_load/parquet/
bronze_autoload(f"{VOLUME_ROOT}/full_load/parquet/products_full_load.parquet", "products_raw")

#Orders
bronze_autoload(f"{VOLUME_ROOT}/full_load/parquet/orders_full_load.parquet", "orders_raw")


#Incremental Auto Loader

In [0]:
def autoload_cdc(subfolder_glob, schema_hint_name, table_name):

    spark.readStream.format("cloudFiles")\
        .option("cloudFiles.format", "parquet")\
        .option("header", True)\
        .option("cloudFiles.schemaLocation", f"{VOLUME_ROOT}/_schema/{schema_hint_name}")\
        .load(f"{VOLUME_ROOT}/incremental_load/{subfolder_glob}")\
        .withColumn("_source_file", F.col("_metadata.file_path"))\
        .withColumn("_ingested_at", F.current_timestamp())\
        .writeStream.format("delta")\
        .option("checkpointLocation", f"{VOLUME_ROOT}/_checkpoint/{schema_hint_name}")\
        .trigger(availableNow=True)\
        .toTable(f"ecommerce_catalog.bronze.{table_name}")\
        .awaitTermination()

autoload_cdc("orders/parquet/orders_incr_*.parquet", "orders_incr", "orders_incr")

autoload_cdc("products/parquet/products_incr_*.parquet", "products_incr", "products_incr")

In [0]:
import os
import shutil
from glob import glob

processed_files = []

#list processed files from the Delta tables
for table in ["orders_incr", "products_incr"]:
    df = spark.read.table(f"ecommerce_catalog.bronze.{table}")
    files = [row[0] for row in df.select("_source_file").distinct().collect()]
    processed_files.extend(files)

target_dir = "/Volumes/ecommerce_catalog/bronze/raw_files/Incremental_processed/"
os.makedirs(target_dir, exist_ok=True)

#delete processed files from the source volume
for file_path in processed_files:
    if os.path.exists(file_path):
        shutil.copy(file_path, os.path.join(target_dir, os.path.basename(file_path)))

